# HearAssist — Test du modèle fine-tuné sur de vrais enregistrements

Transcrit tes propres fichiers audio en darija avec le modèle fine-tuné, et compare avec `whisper-medium` sans entraînement.

**Réglages Kaggle (panneau de droite) :** Accelerator = **GPU T4 x2** · Internet = **On**.

**Input (Add Input) :**
1. **Le modèle** : l'Output du notebook de fine-tuning terminé (onglet *Your Work → Notebooks*). Il doit contenir le dossier `ct2/`.
2. **Tes audios** : un dataset Kaggle avec tes fichiers `.wav`, `.mp3`, `.m4a`, `.ogg` ou `.flac`.
3. *(Facultatif)* Un fichier `references.csv` dans le dataset des audios, avec les colonnes `fichier,texte`, pour calculer le WER sur tes enregistrements.

## 1. Installation

In [ ]:
import os, glob
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!pip install -q faster-whisper jiwer

## 2. Modèle et fichiers audio

In [ ]:
models = sorted(glob.glob("/kaggle/input/**/ct2/model.bin", recursive=True))
assert models, "Modèle introuvable : ajoute en Input l'Output du notebook de fine-tuning (dossier ct2/)"
MODEL_DIR = os.path.dirname(models[0])
print("Modèle :", MODEL_DIR)

EXT = (".wav", ".mp3", ".m4a", ".ogg", ".flac", ".opus", ".webm")
audios = sorted(f for f in glob.glob("/kaggle/input/**/*", recursive=True)
                if f.lower().endswith(EXT) and "/ct2/" not in f)
assert audios, "Aucun fichier audio trouvé : ajoute en Input le dataset contenant tes enregistrements"
print(len(audios), "fichiers audio :")
for f in audios:
    print("  ", os.path.basename(f))

## 3. Chargement des modèles
`COMPARER = True` charge aussi `whisper-medium` sans entraînement, pour voir ce qu'apporte le fine-tuning.

In [ ]:
from faster_whisper import WhisperModel

COMPARER = True
finetune = WhisperModel(MODEL_DIR, device="cuda", compute_type="float16")
base = WhisperModel("medium", device="cuda", compute_type="float16") if COMPARER else None

## 4. Transcription
Le filtre VAD retire les silences, comme dans l'application en temps réel.

In [ ]:
import time
from IPython.display import Audio, display

def transcrire(model, path):
    t0 = time.perf_counter()
    segments, info = model.transcribe(path, language="ar", beam_size=5, vad_filter=True,
                                      condition_on_previous_text=False)
    texte = " ".join(s.text.strip() for s in segments)
    return texte, info.duration, time.perf_counter() - t0

resultats = []
for path in audios:
    nom = os.path.basename(path)
    print("=" * 80)
    print(nom)
    try:
        display(Audio(filename=path))
    except Exception:
        pass  # certains formats ne se lisent pas dans le navigateur
    texte, duree, calcul = transcrire(finetune, path)
    print(f"Fine-tuné ({duree:.1f} s d'audio, {calcul:.1f} s de calcul) :")
    print("  ", texte)
    ligne = {"fichier": nom, "duree_s": round(duree, 1), "finetune": texte}
    if base:
        texte_base, _, _ = transcrire(base, path)
        print("Whisper medium sans entraînement :")
        print("  ", texte_base)
        ligne["base"] = texte_base
    resultats.append(ligne)

## 5. Export des transcriptions

In [ ]:
import pandas as pd
df = pd.DataFrame(resultats)
df.to_csv("/kaggle/working/transcriptions_audios_reels.csv", index=False, encoding="utf-8-sig")
pd.set_option("display.max_colwidth", None)
df

## 6. WER sur tes enregistrements (facultatif)
Nécessite `references.csv` (colonnes `fichier,texte`) : `texte` est ce qui est réellement dit, écrit en arabe.
Avec quelques fichiers seulement, ce WER donne une tendance, pas un score fiable.

In [ ]:
import re, unicodedata, jiwer

refs_csv = glob.glob("/kaggle/input/**/references.csv", recursive=True)
if not refs_csv:
    print("Pas de references.csv : étape ignorée")
else:
    # Même normalisation que text_norm.normalize (scripts du fine-tuning)
    DIAC = re.compile(r"[\u0610-\u061a\u064b-\u065f\u0670\u06d6-\u06ed]")
    def normalize(t):
        t = unicodedata.normalize("NFKC", str(t))
        t = DIAC.sub("", t).replace("\u0640", "")
        t = t.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "0123456789" * 2)).lower()
        t = re.sub("[آأإٱ]", "ا", t).replace("ى", "ي").replace("ة", "ه")
        t = re.sub(r"[^\w\s]|_", " ", t)
        return re.sub(r"\s+", " ", t).strip()

    refs = pd.read_csv(refs_csv[0])
    m = df.merge(refs, on="fichier")
    print(len(m), "fichiers avec référence")
    cols = ["finetune"] + (["base"] if "base" in m else [])
    for col in cols:
        r = [normalize(x) for x in m["texte"]]
        h = [normalize(x) for x in m[col]]
        print(f"{col:9s} | WER {jiwer.wer(r, h):.1%} | CER {jiwer.cer(r, h):.1%}")